# Análisis con las técnicas de las guías
Semana 4: estadística descriptiva y Spearman. Semana 6: H0/H1, nivel de significancia e interpretación del valor p.

P1 (localidad), P2 (tipo de alojamiento) y P5 (anfitriones) son **preguntas descriptivas**, no hipótesis contrastadas. H3 y H4 conservan su numeración para relacionarlas con los KPIs. No se aplican pruebas adicionales a las de las guías.


In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from bi_airbnb import ALPHA, DIR_RESULTADOS, cargar_base, descriptivos, guardar_csv, spearman

todo, d, hosts = cargar_base()


## 1. P1 y P2: precio por localidad y alojamiento
Comparamos n, media, mediana, cuartiles y RIQ (semana 4, pp. 1–3). Se incluyen todos los grupos con datos; un grupo pequeño se interpreta con cautela. Las diferencias observadas no se presentan como diferencias significativas de la población.


In [2]:
for pregunta, grupo in [("P1", "localidad"), ("P2", "tipo_alojamiento")]:
    tabla = descriptivos(d, grupo)
    guardar_csv(tabla, DIR_RESULTADOS, f"descriptivos_{pregunta}.csv")
    display(tabla)


,localidad,n,mediana,media,q1,q3,riq
0,Chapinero,4675,190531.000,224623.341025,148000.0000,276116.5900,128116.5900
1,Usaquén,3271,190000.000,216746.862981,137751.0000,270472.0000,132721.0000
2,Barrios Unidos,1004,148748.850,158210.967231,87931.2925,187981.2500,100049.9575
3,Santa Fe,1395,148453.000,165168.618817,115258.5050,197089.5250,81831.0200
4,Teusaquillo,2732,139000.000,155398.796669,93904.5000,184721.6750,90817.1750
5,Fontibón,978,137259.500,157383.409550,93887.8300,190354.0000,96466.1700
6,La Candelaria,671,135500.000,151362.474724,95200.4000,188245.0000,93044.6000
7,Ciudad Bolívar,38,134205.500,153590.760789,95136.0000,192123.2500,96987.2500
8,Suba,1120,133654.715,153718.891786,80000.0000,194329.1250,114329.1250
9,Puente Aranda,220,130296.000,146568.841909,83372.8500,170685.2500,87312.4000


,tipo_alojamiento,n,mediana,media,q1,q3,riq
0,Habitación de hotel,27,195000.00,198868.564074,130057.5000,234134.500,104077.0000
1,Vivienda completa,13200,179331.69,207469.600180,138327.4975,245000.000,106672.5025
2,Habitación privada,4694,93000.00,117087.145415,68679.5000,139112.695,70433.1950
3,Habitación compartida,77,52180.86,69425.761688,41280.0000,80403.330,39123.3300


## 2. H3 y H4: asociación con el precio
H3: capacidad y precio. H4: calificación y precio. Para cada par, H0: rho de Spearman = 0; H1: rho ≠ 0. Se eliminan solo los nulos del par. Spearman describe la asociación entre rangos (semana 4, pp. 2–3 y 6–7).

Con α = 0,05 se rechaza H0 si p < α; en otro caso no se rechaza. El valor p no es la probabilidad de que H0 sea verdadera. Error tipo I: rechazar una H0 verdadera; tipo II: no rechazar una H0 falsa (semana 6, pp. 2–6).

Los valores p se interpretan por contraste, sin una decisión conjunta sobre ambos. La captura no es una muestra aleatoria y los anuncios de un mismo anfitrión pueden estar relacionados: la inferencia es exploratoria y no demuestra causalidad ni garantiza generalización.


In [3]:
filas = []
for hipotesis, variable in [("H3", "capacidad"), ("H4", "calificacion")]:
    n, rho, p = spearman(d, variable)
    filas.append([hipotesis, variable, "Spearman", n, rho, p])

tests = pd.DataFrame(filas, columns=["hipotesis", "variable", "prueba", "n", "rho", "p"])
tests["decision"] = np.where(tests.p < ALPHA, "Rechazar H0", "No rechazar H0")
tests["p_presentacion"] = tests.p.map(lambda p: "< 0,001" if p < .001 else f"{p:.4f}")
guardar_csv(tests, DIR_RESULTADOS, "hipotesis.csv")
tests[["hipotesis", "variable", "n", "rho", "p_presentacion", "decision"]]


,hipotesis,variable,n,rho,p_presentacion,decision
0,H3,capacidad,17998,0.480350,"< 0,001",Rechazar H0
1,H4,calificacion,14549,0.046257,"< 0,001",Rechazar H0


## 3. P5: comparación descriptiva de anfitriones
Una fila por anfitrión. Comparamos los indicadores de quienes tienen un anuncio y quienes tienen dos o más. Las diferencias de medias o medianas describen esta captura; no se les asigna un valor p ni se declara superioridad de un grupo.


In [4]:
indicadores = ["precio_mediano", "ocupacion_media_dias", "resenas_12m_media", "calificacion_mediana"]
comparacion = hosts.groupby("grupo_h5")[indicadores].agg(["count", "mean", "median"])
comparacion.columns = [f"{indicador}_{medida}" for indicador, medida in comparacion.columns]
comparacion = comparacion.reset_index().rename(columns={"grupo_h5": "grupo_anfitrion"})
guardar_csv(comparacion, DIR_RESULTADOS, "descriptivos_anfitriones.csv")
comparacion


,grupo_anfitrion,precio_mediano_count,precio_mediano_mean,precio_mediano_median,ocupacion_media_dias_count,ocupacion_media_dias_mean,ocupacion_media_dias_median,resenas_12m_media_count,resenas_12m_media_mean,resenas_12m_media_median,calificacion_mediana_count,calificacion_mediana_mean,calificacion_mediana_median
0,1 anuncio,5417,165423.652749,147799.0,5417,54.213771,14.000000,5417,8.420713,2.000000,3973,4.792215,4.89
1,2 o más anuncios,2378,177661.002960,155584.1,2378,62.401363,36.428571,2378,10.968672,4.666667,2198,4.736699,4.83
